# Pipeline de Verificação: Classificador como Filtro Inicial

**Objetivo:** Avaliar um pipeline onde o **classificador de raças** (ConvNeXt-Tiny) atua como **filtro leve** antes do **verificador de identidade** (ConvNeXt-Small + ArcFace).

**Hipótese:** Se o classificador detecta que duas imagens são de raças/macro-grupos claramente diferentes, podemos rejeitar o par imediatamente (identidade diferente) sem rodar o verificador — economizando custo computacional.

**Diferença do experimento de fusão anterior:** Não combinamos scores. O classificador age como um *gate* binário:
- Se `breed_overlap < threshold` → **rejeita direto** (diferentes)
- Caso contrário → **passa para o verificador**, que decide

**Métricas de comparação:**
1. Verificador Solo (baseline)
2. Pipeline: Classificador-filtro → Verificador
3. % de pares filtrados (não precisam do verificador)
4. AUC, Accuracy, F1 por pair_type

---

## 1. Setup e Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Caminhos e Dataset

In [ ]:
import subprocess, time

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_DATA = '/content/local_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# 1) Extrair dataset de verificacao (PetFace)
DATASET_DIR = os.path.join(LOCAL_DATA, 'dogs')
ARCHIVE_PATH = os.path.join(DRIVE_BASE, 'dogs', 'dog.tar.gz')

if not os.path.isdir(DATASET_DIR):
    assert os.path.isfile(ARCHIVE_PATH), f"Arquivo nao encontrado: {ARCHIVE_PATH}"
    os.makedirs(DATASET_DIR, exist_ok=True)
    print("Extraindo dataset de verificacao para SSD local...")
    t0 = time.time()
    subprocess.run(
        f'tar -xzf "{ARCHIVE_PATH}" -C "{DATASET_DIR}"',
        shell=True, check=True
    )
    print(f"Extracao concluida em {time.time()-t0:.1f}s")
else:
    print("Dataset local ja existe, pulando extracao.")

# 2) Copiar pesos do verificador (ArcFace)
VERIFIER_WEIGHTS_DRIVE = os.path.join(DRIVE_BASE, 'dogs', 'best_arcface.pth')
VERIFIER_WEIGHTS = os.path.join(LOCAL_DATA, 'best_arcface.pth')

if not os.path.isfile(VERIFIER_WEIGHTS):
    assert os.path.isfile(VERIFIER_WEIGHTS_DRIVE), f"Pesos nao encontrados: {VERIFIER_WEIGHTS_DRIVE}"
    print("Copiando pesos do verificador para SSD local...")
    subprocess.run(f'cp "{VERIFIER_WEIGHTS_DRIVE}" "{VERIFIER_WEIGHTS}"', shell=True, check=True)
    print("Copia concluida.")
else:
    print("Pesos do verificador ja existem localmente.")

# Pesos e configuracoes do classificador (lidos direto do Drive)
CLASSIFIER_WEIGHTS = f"{DRIVE_BASE}/classification_exp/results/best_model_stanford.pth"
LABEL_ENCODER_PATH = f"{DRIVE_BASE}/classification_exp/results/label_encoder_stanford.json"
MACRO_GROUPS_PATH = f"{DRIVE_BASE}/classification_exp/results/breed_macro_groups.json"
VERIF_CSV = f"{DRIVE_BASE}/dogs/split/verification_test_pairs.csv"

RESULTS_DIR = f"{DRIVE_BASE}/classification_exp/results/classifier_filter"
os.makedirs(RESULTS_DIR, exist_ok=True)

df_verif = pd.read_csv(VERIF_CSV)
print(f"Pares de verificação: {len(df_verif)}")
print(df_verif['pair_type'].value_counts())

## 3. Definição dos Modelos

In [ ]:
# Verificador (ConvNeXt-Small + ArcFace, embedding 512-d)
class EmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model('convnext_small.fb_in22k_ft_in1k', pretrained=pretrained, num_classes=0)
        self.embedding = nn.Sequential(
            nn.Linear(self.backbone.num_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )
    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding(features)
        return F.normalize(embeddings, p=2, dim=1)

# Classificador (ConvNeXt-Tiny, 120 classes)
def create_classifier(num_classes):
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

## 4. Carregar Modelos e Macro-Grupos

In [ ]:
# Label Encoder
with open(LABEL_ENCODER_PATH, 'r') as f:
    label_encoder = json.load(f)
num_classes = label_encoder['num_classes']
idx_to_label = {int(k): v for k, v in label_encoder['idx_to_label'].items()}

# Macro-Grupos
with open(MACRO_GROUPS_PATH, 'r') as f:
    macro_data = json.load(f)
breed_to_group = macro_data['breed_to_group']

# Classificador
classifier = create_classifier(num_classes)
clf_state = torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)
clf_state = clf_state.get('model_state_dict', clf_state)
clf_state = {k.replace('_orig_mod.', '').replace('module.', ''): v for k, v in clf_state.items()}
classifier.load_state_dict(clf_state)
classifier.to(DEVICE).eval()
print(f"Classificador carregado ({num_classes} classes).")

# Verificador
verifier = EmbeddingModel(embedding_dim=512, pretrained=False)
verif_ckpt = torch.load(VERIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)
verifier.load_state_dict(verif_ckpt['backbone'])
verifier.to(DEVICE).eval()
print("Verificador carregado.")

## 5. Dataset e Inferência Dupla

In [ ]:
eval_transform = T.Compose([
    T.Resize((224, 224), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PairDataset(Dataset):
    def __init__(self, df, root_dir, transform):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img1 = Image.open(os.path.join(self.root_dir, row['filename1'])).convert('RGB')
        img2 = Image.open(os.path.join(self.root_dir, row['filename2'])).convert('RGB')
        t1 = self.transform(img1)
        t2 = self.transform(img2)
        return t1, t2, int(row['label']), row.get('pair_type', 'unknown')

test_loader = DataLoader(PairDataset(df_verif, DATASET_DIR, eval_transform),
                         batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# Inferência: captura scores do verificador + probabilidades do classificador
all_labels = []
all_pair_types = []
sims_verifier = []          # Cosine similarity normalizada [0,1]
breed_overlaps = []         # Dot product das probabilidades softmax
top1_match = []             # Top-1 raça bate?
top1_group_match = []       # Top-1 macro-grupo bate?
clf_probs_1 = []            # Softmax completa da img1
clf_probs_2 = []            # Softmax completa da img2

print("Executando inferência dupla...")
with torch.no_grad():
    for img1, img2, lbl, ptype in tqdm(test_loader):
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        all_labels.extend(lbl.numpy())
        all_pair_types.extend(ptype)

        with torch.amp.autocast('cuda'):
            # Verificador
            emb1 = verifier(img1)
            emb2 = verifier(img2)
            cos_sim = F.cosine_similarity(emb1, emb2, dim=1)
            norm_sim = (cos_sim + 1.0) / 2.0
            sims_verifier.extend(norm_sim.cpu().numpy())

            # Classificador
            probs1 = torch.softmax(classifier(img1), dim=1)
            probs2 = torch.softmax(classifier(img2), dim=1)

            # Breed overlap (dot product das probs)
            overlap = torch.sum(probs1 * probs2, dim=1)
            breed_overlaps.extend(overlap.cpu().numpy())

            # Top-1 match
            top1_1 = probs1.argmax(dim=1)
            top1_2 = probs2.argmax(dim=1)
            top1_match.extend((top1_1 == top1_2).cpu().numpy())

            # Top-1 macro-group match
            for i in range(len(lbl)):
                g1 = breed_to_group.get(idx_to_label.get(top1_1[i].item(), ''), 'Unknown')
                g2 = breed_to_group.get(idx_to_label.get(top1_2[i].item(), ''), 'Unknown')
                top1_group_match.append(g1 == g2)

            clf_probs_1.append(probs1.cpu().numpy())
            clf_probs_2.append(probs2.cpu().numpy())

all_labels = np.array(all_labels)
sims_verifier = np.array(sims_verifier)
breed_overlaps = np.array(breed_overlaps)
top1_match = np.array(top1_match)
top1_group_match = np.array(top1_group_match)
all_pair_types = np.array(all_pair_types)
clf_probs_1 = np.concatenate(clf_probs_1, axis=0)
clf_probs_2 = np.concatenate(clf_probs_2, axis=0)

print(f"Inferência concluída! {len(all_labels)} pares.")

## 6. Funções de Avaliação

In [ ]:
def evaluate(scores, labels, pair_types, name="Strategy"):
    """Avalia uma estratégia dado scores contínuos e labels binários."""
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thresh = thresholds[best_idx]
    preds = (scores >= best_thresh).astype(int)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)

    # Hard negatives
    mask_hard = (pair_types == 'negative_same_breed')
    acc_hard = accuracy_score(labels[mask_hard], preds[mask_hard]) if mask_hard.sum() > 0 else 0

    # Easy negatives
    mask_easy = (pair_types == 'negative_random')
    acc_easy = accuracy_score(labels[mask_easy], preds[mask_easy]) if mask_easy.sum() > 0 else 0

    print(f"--- {name} ---")
    print(f"AUC: {auc:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"Thresh (Youden): {best_thresh:.4f}")
    print(f"Acc Hard Neg (same breed): {acc_hard:.4f} ({mask_hard.sum()} pares)")
    print(f"Acc Easy Neg (random):     {acc_easy:.4f} ({mask_easy.sum()} pares)")
    print()
    return {'name': name, 'auc': auc, 'acc': acc, 'f1': f1, 'prec': prec, 'rec': rec,
            'thresh': best_thresh, 'fpr': fpr, 'tpr': tpr, 'preds': preds,
            'acc_hard': acc_hard, 'acc_easy': acc_easy}


def evaluate_binary(preds, labels, pair_types, name="Strategy"):
    """Avalia dado predições binárias já decididas."""
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)

    mask_hard = (pair_types == 'negative_same_breed')
    acc_hard = accuracy_score(labels[mask_hard], preds[mask_hard]) if mask_hard.sum() > 0 else 0
    mask_easy = (pair_types == 'negative_random')
    acc_easy = accuracy_score(labels[mask_easy], preds[mask_easy]) if mask_easy.sum() > 0 else 0

    print(f"--- {name} ---")
    print(f"Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"Acc Hard Neg (same breed): {acc_hard:.4f}")
    print(f"Acc Easy Neg (random):     {acc_easy:.4f}")
    print()
    return {'name': name, 'acc': acc, 'f1': f1, 'prec': prec, 'rec': rec,
            'acc_hard': acc_hard, 'acc_easy': acc_easy}

print("Funções de avaliação prontas.")

## 7. Baseline: Verificador Solo

In [ ]:
res_baseline = evaluate(sims_verifier, all_labels, all_pair_types,
                        "Verificador Solo (ConvNeXt-S ArcFace)")
verif_thresh = res_baseline['thresh']

## 8. Pipeline: Classificador como Filtro

A ideia é:
1. O classificador analisa ambas as imagens (custo baixo — ConvNeXt-Tiny)
2. Se detecta que são raças/grupos **claramente diferentes** → rejeita (prediz 0 = identidade diferente)
3. Caso contrário → passa para o verificador (custo alto — ConvNeXt-Small)

Testamos 3 tipos de critério para o filtro:
- **Breed Overlap**: Dot product softmax < threshold → rejeita
- **Top-1 Breed**: Se top-1 da raça é diferente → rejeita
- **Top-1 Macro-Grupo**: Se top-1 do macro-grupo é diferente → rejeita

In [ ]:
def pipeline_filter(filter_mask, filter_name, verif_scores, verif_threshold, labels, pair_types):
    """
    Pipeline: classificador como filtro.

    Args:
        filter_mask: bool array. True = filtrado pelo classificador (rejeitado, pred=0)
        filter_name: nome da estratégia
        verif_scores: scores do verificador (para os não-filtrados)
        verif_threshold: threshold ótimo do verificador
        labels: ground truth
        pair_types: tipos de par
    """
    n_total = len(labels)
    n_filtered = filter_mask.sum()
    n_pass = n_total - n_filtered
    pct_filtered = n_filtered / n_total * 100

    # Pares filtrados: predição = 0 (diferentes)
    # Pares passantes: predição baseada no verificador
    preds = np.zeros(n_total, dtype=int)
    preds[~filter_mask] = (verif_scores[~filter_mask] >= verif_threshold).astype(int)

    # Métricas
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)

    # Análise dos filtrados
    filtered_labels = labels[filter_mask]
    n_correct_filter = (filtered_labels == 0).sum()  # corretamente rejeitados
    n_wrong_filter = (filtered_labels == 1).sum()     # positivos perdidos (falso negativo)

    # Hard negatives
    mask_hard = (pair_types == 'negative_same_breed')
    acc_hard = accuracy_score(labels[mask_hard], preds[mask_hard]) if mask_hard.sum() > 0 else 0

    # Easy negatives
    mask_easy = (pair_types == 'negative_random')
    acc_easy = accuracy_score(labels[mask_easy], preds[mask_easy]) if mask_easy.sum() > 0 else 0

    # Positivos
    mask_pos = (pair_types == 'positive')
    acc_pos = accuracy_score(labels[mask_pos], preds[mask_pos]) if mask_pos.sum() > 0 else 0

    print(f"{'='*70}")
    print(f"PIPELINE: {filter_name}")
    print(f"{'='*70}")
    print(f"\nDistribuição do filtro:")
    print(f"  Total de pares:       {n_total:>6}")
    print(f"  Filtrados (rejeitados):{n_filtered:>6} ({pct_filtered:.1f}%)")
    print(f"  Passam p/ verificador: {n_pass:>6} ({100-pct_filtered:.1f}%)")
    print(f"\nAnálise dos filtrados:")
    print(f"  Corretamente rejeitados (TN): {n_correct_filter:>6}")
    print(f"  Positivos perdidos (FN):      {n_wrong_filter:>6}")
    if n_filtered > 0:
        print(f"  Precisão do filtro:           {n_correct_filter/n_filtered:.4f}")
    print(f"\nMétricas Gerais:")
    print(f"  Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"\nPor Tipo de Par:")
    print(f"  Acc Positivos:       {acc_pos:.4f}")
    print(f"  Acc Hard Neg:        {acc_hard:.4f}")
    print(f"  Acc Easy Neg:        {acc_easy:.4f}")
    print()

    return {
        'name': filter_name, 'acc': acc, 'f1': f1, 'prec': prec, 'rec': rec,
        'n_filtered': int(n_filtered), 'pct_filtered': pct_filtered,
        'n_correct_filter': int(n_correct_filter), 'n_wrong_filter': int(n_wrong_filter),
        'acc_hard': acc_hard, 'acc_easy': acc_easy, 'acc_pos': acc_pos,
        'preds': preds
    }

### 8a. Filtro por Top-1 Raça Diferente

In [ ]:
# Se top-1 da raça predita é diferente → rejeita
filter_top1_breed = ~top1_match  # True = raças diferentes → filtrar

res_top1_breed = pipeline_filter(
    filter_top1_breed, "Top-1 Raça Diferente → Rejeita",
    sims_verifier, verif_thresh, all_labels, all_pair_types
)

### 8b. Filtro por Top-1 Macro-Grupo Diferente

In [ ]:
# Se top-1 do macro-grupo predito é diferente → rejeita
filter_top1_group = ~top1_group_match  # True = grupos diferentes → filtrar

res_top1_group = pipeline_filter(
    filter_top1_group, "Top-1 Macro-Grupo Diferente → Rejeita",
    sims_verifier, verif_thresh, all_labels, all_pair_types
)

### 8c. Filtro por Breed Overlap (Grid Search de Threshold)

In [ ]:
# Grid Search: diferentes thresholds de breed overlap
overlap_thresholds = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]

results_overlap = []
for ot in overlap_thresholds:
    filter_mask = (breed_overlaps < ot)
    res = pipeline_filter(
        filter_mask, f"Breed Overlap < {ot}",
        sims_verifier, verif_thresh, all_labels, all_pair_types
    )
    res['overlap_threshold'] = ot
    results_overlap.append(res)

## 9. Tabela Comparativa

In [ ]:
# Montar tabela comparativa
rows = []

# Baseline
rows.append({
    'Estratégia': 'Verificador Solo',
    'AUC': res_baseline['auc'],
    'Acc': res_baseline['acc'],
    'F1': res_baseline['f1'],
    'Prec': res_baseline['prec'],
    'Rec': res_baseline['rec'],
    '% Filtrado': 0.0,
    'Acc Hard': res_baseline['acc_hard'],
    'Acc Easy': res_baseline['acc_easy'],
})

# Top-1 Breed
rows.append({
    'Estratégia': 'Filtro: Top-1 Raça Dif.',
    'AUC': '-',
    'Acc': res_top1_breed['acc'],
    'F1': res_top1_breed['f1'],
    'Prec': res_top1_breed['prec'],
    'Rec': res_top1_breed['rec'],
    '% Filtrado': res_top1_breed['pct_filtered'],
    'Acc Hard': res_top1_breed['acc_hard'],
    'Acc Easy': res_top1_breed['acc_easy'],
})

# Top-1 Group
rows.append({
    'Estratégia': 'Filtro: Top-1 Grupo Dif.',
    'AUC': '-',
    'Acc': res_top1_group['acc'],
    'F1': res_top1_group['f1'],
    'Prec': res_top1_group['prec'],
    'Rec': res_top1_group['rec'],
    '% Filtrado': res_top1_group['pct_filtered'],
    'Acc Hard': res_top1_group['acc_hard'],
    'Acc Easy': res_top1_group['acc_easy'],
})

# Overlap thresholds
for r in results_overlap:
    rows.append({
        'Estratégia': f"Filtro: Overlap<{r['overlap_threshold']}",
        'AUC': '-',
        'Acc': r['acc'],
        'F1': r['f1'],
        'Prec': r['prec'],
        'Rec': r['rec'],
        '% Filtrado': r['pct_filtered'],
        'Acc Hard': r['acc_hard'],
        'Acc Easy': r['acc_easy'],
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 10. Visualizações

In [ ]:
# Gráfico 1: % Filtrado vs Accuracy vs F1 por threshold de overlap
fig, ax1 = plt.subplots(figsize=(12, 6))

ots = [r['overlap_threshold'] for r in results_overlap]
accs = [r['acc'] for r in results_overlap]
f1s = [r['f1'] for r in results_overlap]
pcts = [r['pct_filtered'] for r in results_overlap]

ax1.set_xlabel('Breed Overlap Threshold', fontsize=12)
ax1.set_ylabel('Accuracy / F1', fontsize=12, color='tab:blue')
ax1.plot(ots, accs, 'b-o', label='Accuracy', linewidth=2)
ax1.plot(ots, f1s, 'g-s', label='F1', linewidth=2)
ax1.axhline(y=res_baseline['acc'], color='b', linestyle='--', alpha=0.4, label=f"Baseline Acc={res_baseline['acc']:.4f}")
ax1.axhline(y=res_baseline['f1'], color='g', linestyle='--', alpha=0.4, label=f"Baseline F1={res_baseline['f1']:.4f}")
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_ylim(0.5, 1.0)

ax2 = ax1.twinx()
ax2.set_ylabel('% Pares Filtrados', fontsize=12, color='tab:red')
ax2.bar(ots, pcts, alpha=0.3, color='red', width=0.02, label='% Filtrado')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.set_ylim(0, 100)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=9)

plt.title('Pipeline Classificador→Verificador: Trade-off Filtragem vs Performance',
          fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'tradeoff_filtragem_performance.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Salvo: {fig_path}")
plt.show()

In [ ]:
# Gráfico 2: Comparação das estratégias
strategies = ['Verificador Solo', 'Top-1 Raça', 'Top-1 Grupo']
acc_vals = [res_baseline['acc'], res_top1_breed['acc'], res_top1_group['acc']]
f1_vals = [res_baseline['f1'], res_top1_breed['f1'], res_top1_group['f1']]
pct_vals = [0, res_top1_breed['pct_filtered'], res_top1_group['pct_filtered']]

# Adicionar melhor overlap
best_overlap = max(results_overlap, key=lambda r: r['f1'])
strategies.append(f"Overlap<{best_overlap['overlap_threshold']}")
acc_vals.append(best_overlap['acc'])
f1_vals.append(best_overlap['f1'])
pct_vals.append(best_overlap['pct_filtered'])

x = np.arange(len(strategies))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width, acc_vals, width, label='Accuracy', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x, f1_vals, width, label='F1 Score', color='#55A868', alpha=0.85)
bars3 = ax.bar(x + width, [p/100 for p in pct_vals], width, label='Fração Filtrada', color='#C44E52', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Comparação: Verificador Solo vs Pipeline com Filtro',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strategies, fontsize=10)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'comparacao_estrategias.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Salvo: {fig_path}")
plt.show()

In [ ]:
# Gráfico 3: Distribuição do breed_overlap por tipo de par
fig, ax = plt.subplots(figsize=(12, 6))

for ptype in ['positive', 'negative_same_breed', 'negative_random']:
    mask = (all_pair_types == ptype)
    ax.hist(breed_overlaps[mask], bins=100, alpha=0.6, label=ptype, density=True)

ax.set_xlabel('Breed Overlap (dot product softmax)', fontsize=12)
ax.set_ylabel('Densidade', fontsize=12)
ax.set_title('Distribuição do Breed Overlap por Tipo de Par', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'distribuicao_breed_overlap.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Salvo: {fig_path}")
plt.show()

## 11. Análise de Custo Computacional

In [ ]:
# Análise comparativa de custo
# ConvNeXt-Tiny: ~28.6M params
# ConvNeXt-Small: ~50.2M params
clf_params = sum(p.numel() for p in classifier.parameters())
verif_params = sum(p.numel() for p in verifier.parameters())

print("='*70")
print("ANÁLISE DE CUSTO COMPUTACIONAL")
print("='*70")
print(f"\nClassificador (ConvNeXt-Tiny):  {clf_params/1e6:.1f}M params")
print(f"Verificador   (ConvNeXt-Small): {verif_params/1e6:.1f}M params")
print(f"Razão: Verificador é {verif_params/clf_params:.1f}x maior que o Classificador")

print(f"\n--- Cenário: Solo Verificador ---")
print(f"  Cada par requer: 2 forward passes do Verificador")
print(f"  Custo total por par: 2 × {verif_params/1e6:.1f}M = {2*verif_params/1e6:.1f}M params processados")

for res in [res_top1_breed, res_top1_group, best_overlap]:
    pct_f = res['pct_filtered']
    pct_pass = 100 - pct_f
    # Custo pipeline: TODOS os pares passam pelo classificador (2 forward passes)
    # + Apenas os não-filtrados passam pelo verificador (2 forward passes)
    cost_clf = 2 * clf_params  # sempre
    cost_verif = 2 * verif_params * (pct_pass / 100)  # apenas % que passa
    cost_total = cost_clf + cost_verif
    cost_baseline = 2 * verif_params
    savings = (1 - cost_total / cost_baseline) * 100

    print(f"\n--- Cenário: {res['name']} ---")
    print(f"  Filtrados: {pct_f:.1f}% | Passam: {pct_pass:.1f}%")
    print(f"  Custo classificador (todos):     {2*clf_params/1e6:.1f}M")
    print(f"  Custo verificador ({pct_pass:.1f}% pares): {cost_verif/1e6:.1f}M")
    print(f"  Custo total por par:             {cost_total/1e6:.1f}M")
    print(f"  Economia vs baseline:            {savings:+.1f}%")
    print(f"  Perda de Acc:                    {res['acc'] - res_baseline['acc']:+.4f}")
    print(f"  Perda de F1:                     {res['f1'] - res_baseline['f1']:+.4f}")

## 12. Salvar Resultados

In [ ]:
# Salvar tabela comparativa
csv_path = os.path.join(RESULTS_DIR, 'resultados_pipeline_filtro.csv')
df_results.to_csv(csv_path, index=False, float_format='%.4f')
print(f"Tabela salva em: {csv_path}")

# Salvar JSON detalhado
detailed = {
    'baseline': {k: v for k, v in res_baseline.items() if k not in ['fpr', 'tpr', 'preds']},
    'top1_breed': {k: v for k, v in res_top1_breed.items() if k != 'preds'},
    'top1_group': {k: v for k, v in res_top1_group.items() if k != 'preds'},
    'overlap_thresholds': [{k: v for k, v in r.items() if k != 'preds'} for r in results_overlap],
}
json_path = os.path.join(RESULTS_DIR, 'resultados_detalhados.json')
with open(json_path, 'w') as f:
    json.dump(detailed, f, indent=2, default=str)
print(f"Detalhes salvos em: {json_path}")

print(f"\nTodos os resultados em: {RESULTS_DIR}")

## 13. Conclusão

Este notebook avalia se o classificador de raças (ConvNeXt-Tiny, menor e mais rápido) pode servir como filtro inicial no pipeline de verificação de identidade, rejeitando pares de imagens que são claramente de raças/grupos diferentes antes de rodar o verificador (ConvNeXt-Small + ArcFace, maior e mais custoso).

A análise compara:
- **Performance** (Acc, F1, Precision, Recall) do pipeline vs verificador solo
- **Eficiência** (% de pares que não precisam do verificador)
- **Robustez** em diferentes cenários (hard negatives, easy negatives)

O objetivo é encontrar o ponto ótimo onde a perda de acurácia é mínima mas a economia computacional justifica o uso do classificador como filtro.